# 3/3 - Post-processing EDA: Descriptive Multivariate Outlier Detection

Notebook ini **tidak lagi memakai ANOVA dan IG/MI**. ANOVA dan IG/MI sudah menjadi tanggung jawab notebook 2.

Fokus notebook ini adalah deskripsi outlier berdasarkan **final feature set** dari notebook 2, yaitu:

```text
Core Stable Features + PC_Residual dari Hybrid PCA
```

Pertanyaan utama notebook ini:
1. Sampel mana yang masih tampak ekstrem setelah fitur diseleksi dan direduksi?
2. Apakah outlier muncul relatif terhadap kelasnya sendiri?
3. Kelas mana yang memiliki variasi visual ekstrem lebih tinggi?
4. Apakah outlier dekat dengan kelas lain pada ruang fitur final?

Output utama:
- `descriptive_multivariate_outliers.csv`
- `descriptive_multivariate_outlier_summary_by_class.csv`
- `descriptive_multivariate_outlier_method_summary.csv`


In [ ]:

import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet, EmpiricalCovariance
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_samples, silhouette_score
from scipy.stats import chi2

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 120

FINAL_FEATURE_CSV = Path("final_feature_set_for_multivariate_outlier.csv")
FALLBACK_FINAL_FEATURE_CSV = Path("hybrid_pca_final_feature_matrix.csv")
FINAL_MANIFEST_CSV = Path("dataset_final_manifest.csv")

RANDOM_STATE = 42
OUTLIER_QUANTILE = 0.95
MIN_CLASS_SAMPLES = 10


## 1. Load final feature set dari notebook 2

Notebook ini membaca file sample-level yang dibuat oleh notebook 2. File tersebut harus berisi kolom `path`, `class`, dan fitur numerik final.

In [ ]:

def load_final_feature_set():
    if FINAL_FEATURE_CSV.exists():
        feature_path = FINAL_FEATURE_CSV
    elif FALLBACK_FINAL_FEATURE_CSV.exists():
        feature_path = FALLBACK_FINAL_FEATURE_CSV
    else:
        raise FileNotFoundError(
            "Final feature set tidak ditemukan. Jalankan notebook 2 terlebih dahulu hingga menghasilkan "
            "final_feature_set_for_multivariate_outlier.csv."
        )

    df = pd.read_csv(feature_path)
    required = {"path", "class"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"File {feature_path} tidak memiliki kolom wajib: {missing}")

    # Validasi opsional terhadap manifest final.
    if FINAL_MANIFEST_CSV.exists():
        manifest = pd.read_csv(FINAL_MANIFEST_CSV)
        print(f"Manifest final ditemukan: {len(manifest)} sampel")
        if len(manifest) != len(df):
            print(
                "⚠️ Jumlah final feature set berbeda dari manifest. "
                f"Feature set={len(df)}, manifest={len(manifest)}. Cek kembali urutan eksekusi notebook."
            )

    numeric_cols = [
        c for c in df.columns
        if c not in ["path", "class", "filename", "original_path"]
        and pd.api.types.is_numeric_dtype(df[c])
    ]
    if not numeric_cols:
        raise ValueError("Tidak ada fitur numerik final yang dapat dipakai untuk outlier multivariat.")

    print(f"✅ Final feature set dimuat dari: {feature_path}")
    print(f"Jumlah sampel: {len(df)}")
    print(f"Jumlah final features: {len(numeric_cols)}")
    display(df["class"].value_counts().rename_axis("class").reset_index(name="count"))
    display(pd.DataFrame({"final_feature": numeric_cols}))
    return df, numeric_cols


df_final, final_feature_cols = load_final_feature_set()


## 2. Standardisasi fitur final

Semua metode outlier multivariat bekerja pada fitur yang sudah distandardisasi agar skala fitur tidak mendominasi jarak atau density.

In [ ]:

def prepare_scaled_features(df, feature_cols):
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
    X = X.apply(lambda s: s.fillna(s.median()), axis=0)

    # Buang fitur konstan bila ada.
    non_constant_cols = X.columns[X.std(axis=0) > 0].tolist()
    X = X[non_constant_cols]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled_df = pd.DataFrame(X_scaled, columns=non_constant_cols, index=df.index)
    return X_scaled_df, scaler

X_scaled_df, final_scaler = prepare_scaled_features(df_final, final_feature_cols)
print("Fitur final yang dipakai setelah validasi non-konstan:")
display(pd.DataFrame({"feature": X_scaled_df.columns.tolist()}))


## 3. Fungsi outlier multivariat berbasis kelas

Outlier dihitung relatif terhadap kelasnya sendiri. Ini lebih tepat untuk dataset rempah karena setiap kelas memiliki distribusi visual yang berbeda.

In [ ]:

def _quantile_flag(values, q=0.95):
    threshold = np.quantile(values, q)
    return values >= threshold, threshold


def compute_classwise_robust_mahalanobis(df, X_scaled, class_col="class", alpha=0.975):
    scores = pd.Series(index=df.index, dtype=float)
    flags = pd.Series(False, index=df.index, dtype=bool)
    thresholds = {}

    p = X_scaled.shape[1]
    chi_threshold = chi2.ppf(alpha, df=max(p, 1))

    for cls, idx in df.groupby(class_col).groups.items():
        idx = list(idx)
        Xg = X_scaled.loc[idx].values
        if len(idx) < max(MIN_CLASS_SAMPLES, p + 3):
            # Jika sampel terlalu sedikit, pakai empirical covariance sebagai fallback.
            estimator = EmpiricalCovariance()
        else:
            estimator = MinCovDet(random_state=RANDOM_STATE, support_fraction=0.75)
        try:
            estimator.fit(Xg)
            dist = estimator.mahalanobis(Xg)
        except Exception:
            # Fallback aman: jarak Euclidean terhadap centroid.
            center = np.nanmean(Xg, axis=0)
            dist = np.sum((Xg - center) ** 2, axis=1)

        scores.loc[idx] = dist
        flags.loc[idx] = dist >= chi_threshold
        thresholds[cls] = chi_threshold

    return scores, flags.astype(int), thresholds


def compute_classwise_lof(df, X_scaled, class_col="class", q=0.95):
    scores = pd.Series(index=df.index, dtype=float)
    flags = pd.Series(False, index=df.index, dtype=bool)
    thresholds = {}

    for cls, idx in df.groupby(class_col).groups.items():
        idx = list(idx)
        Xg = X_scaled.loc[idx].values
        if len(idx) < 5:
            scores.loc[idx] = 0.0
            flags.loc[idx] = False
            thresholds[cls] = np.nan
            continue

        n_neighbors = min(20, max(2, len(idx) - 1))
        lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination="auto")
        lof.fit_predict(Xg)
        # Semakin besar score, semakin outlier.
        outlier_score = -lof.negative_outlier_factor_
        flag, threshold = _quantile_flag(outlier_score, q=q)

        scores.loc[idx] = outlier_score
        flags.loc[idx] = flag
        thresholds[cls] = threshold

    return scores, flags.astype(int), thresholds


def compute_classwise_isolation_forest(df, X_scaled, class_col="class", contamination=0.05):
    scores = pd.Series(index=df.index, dtype=float)
    flags = pd.Series(False, index=df.index, dtype=bool)
    thresholds = {}

    for cls, idx in df.groupby(class_col).groups.items():
        idx = list(idx)
        Xg = X_scaled.loc[idx].values
        if len(idx) < 10:
            scores.loc[idx] = 0.0
            flags.loc[idx] = False
            thresholds[cls] = np.nan
            continue

        iso = IsolationForest(
            n_estimators=200,
            contamination=contamination,
            random_state=RANDOM_STATE,
        )
        pred = iso.fit_predict(Xg)
        # score_samples: semakin kecil semakin outlier. Balik agar semakin besar semakin outlier.
        outlier_score = -iso.score_samples(Xg)
        flags.loc[idx] = pred == -1
        scores.loc[idx] = outlier_score
        thresholds[cls] = np.quantile(outlier_score, 1 - contamination)

    return scores, flags.astype(int), thresholds


def compute_classwise_pca_reconstruction_error(df, X_scaled, class_col="class", q=0.95):
    errors = pd.Series(index=df.index, dtype=float)
    flags = pd.Series(False, index=df.index, dtype=bool)
    thresholds = {}
    p = X_scaled.shape[1]

    for cls, idx in df.groupby(class_col).groups.items():
        idx = list(idx)
        Xg = X_scaled.loc[idx].values
        if len(idx) < 5 or p < 2:
            # Fallback jarak terhadap centroid.
            center = np.nanmean(Xg, axis=0)
            err = np.sum((Xg - center) ** 2, axis=1)
        else:
            # Simpan sebagian besar dimensi, tetapi tetap sisakan ruang untuk reconstruction error.
            n_components = min(max(1, int(np.ceil(p * 0.70))), p - 1, len(idx) - 1)
            pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
            Z = pca.fit_transform(Xg)
            X_rec = pca.inverse_transform(Z)
            err = np.mean((Xg - X_rec) ** 2, axis=1)

        flag, threshold = _quantile_flag(err, q=q)
        errors.loc[idx] = err
        flags.loc[idx] = flag
        thresholds[cls] = threshold

    return errors, flags.astype(int), thresholds


## 4. Hitung consensus descriptive outlier score

Empat metode dipakai sebagai sudut pandang yang saling melengkapi:

1. **Robust Mahalanobis Distance** untuk jarak multivariat terhadap pusat kelas.
2. **Local Outlier Factor** untuk outlier berbasis density lokal.
3. **Isolation Forest** untuk sampel yang mudah diisolasi pada ruang fitur.
4. **PCA Reconstruction Error** untuk sampel yang tidak mengikuti struktur utama fitur final.

Skor akhir bersifat deskriptif:

```text
0 = normal
1 = indikasi lemah
2 = indikasi sedang
3-4 = indikasi kuat
```


In [ ]:

df_out = df_final.copy()

mah_score, mah_flag, mah_thresholds = compute_classwise_robust_mahalanobis(df_final, X_scaled_df)
lof_score, lof_flag, lof_thresholds = compute_classwise_lof(df_final, X_scaled_df, q=OUTLIER_QUANTILE)
iso_score, iso_flag, iso_thresholds = compute_classwise_isolation_forest(df_final, X_scaled_df, contamination=1 - OUTLIER_QUANTILE)
pca_score, pca_flag, pca_thresholds = compute_classwise_pca_reconstruction_error(df_final, X_scaled_df, q=OUTLIER_QUANTILE)

df_out["mahalanobis_score"] = mah_score
df_out["lof_score"] = lof_score
df_out["isolation_forest_score"] = iso_score
df_out["pca_reconstruction_error"] = pca_score

df_out["flag_mahalanobis"] = mah_flag
df_out["flag_lof"] = lof_flag
df_out["flag_isolation_forest"] = iso_flag
df_out["flag_pca_reconstruction"] = pca_flag

flag_cols = ["flag_mahalanobis", "flag_lof", "flag_isolation_forest", "flag_pca_reconstruction"]
df_out["consensus_multivariate_outlier_score"] = df_out[flag_cols].sum(axis=1)

def label_outlier_strength(score):
    if score >= 3:
        return "strong_multivariate_outlier"
    if score == 2:
        return "moderate_multivariate_outlier"
    if score == 1:
        return "weak_single_method_outlier"
    return "normal"

df_out["outlier_strength"] = df_out["consensus_multivariate_outlier_score"].apply(label_outlier_strength)

display(df_out[["path", "class", "consensus_multivariate_outlier_score", "outlier_strength"] + flag_cols].sort_values(
    ["consensus_multivariate_outlier_score", "class"], ascending=[False, True]
).head(30))

df_out.to_csv("descriptive_multivariate_outliers.csv", index=False)
print("✅ descriptive_multivariate_outliers.csv disimpan")


## 5. Ringkasan outlier per kelas dan per metode

In [ ]:

summary_by_class = (
    df_out.groupby("class")
    .agg(
        n_samples=("path", "count"),
        mean_consensus_score=("consensus_multivariate_outlier_score", "mean"),
        max_consensus_score=("consensus_multivariate_outlier_score", "max"),
        n_weak_or_more=("consensus_multivariate_outlier_score", lambda s: int((s >= 1).sum())),
        n_moderate_or_more=("consensus_multivariate_outlier_score", lambda s: int((s >= 2).sum())),
        n_strong=("consensus_multivariate_outlier_score", lambda s: int((s >= 3).sum())),
    )
    .reset_index()
)
summary_by_class["ratio_moderate_or_more"] = summary_by_class["n_moderate_or_more"] / summary_by_class["n_samples"]
summary_by_class = summary_by_class.sort_values(["ratio_moderate_or_more", "mean_consensus_score"], ascending=False)

display(summary_by_class)
summary_by_class.to_csv("descriptive_multivariate_outlier_summary_by_class.csv", index=False)

method_summary = pd.DataFrame({
    "method": flag_cols,
    "n_flagged": [int(df_out[c].sum()) for c in flag_cols],
})
method_summary["ratio_flagged"] = method_summary["n_flagged"] / len(df_out)
display(method_summary)
method_summary.to_csv("descriptive_multivariate_outlier_method_summary.csv", index=False)

plt.figure(figsize=(12, 5))
sns.barplot(data=summary_by_class, x="class", y="ratio_moderate_or_more")
plt.xticks(rotation=45, ha="right")
plt.title("Rasio Moderate/Strong Multivariate Outlier per Kelas")
plt.xlabel("Kelas")
plt.ylabel("Rasio sampel dengan consensus score ≥ 2")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(data=method_summary, x="method", y="ratio_flagged")
plt.xticks(rotation=30, ha="right")
plt.title("Rasio Sampel yang Ditandai oleh Setiap Metode Outlier")
plt.xlabel("Metode")
plt.ylabel("Rasio flagged")
plt.tight_layout()
plt.show()


## 6. Proyeksi 2D fitur final dengan penanda outlier

Proyeksi ini bukan untuk seleksi fitur, melainkan untuk membaca posisi outlier pada ruang fitur final.

In [ ]:

pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca_2d.fit_transform(X_scaled_df)

df_vis = df_out.copy()
df_vis["PCA_Final_1"] = X_2d[:, 0]
df_vis["PCA_Final_2"] = X_2d[:, 1]
df_vis["is_moderate_or_strong"] = df_vis["consensus_multivariate_outlier_score"] >= 2

plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=df_vis,
    x="PCA_Final_1",
    y="PCA_Final_2",
    hue="class",
    style="is_moderate_or_strong",
    alpha=0.8,
    s=60,
)
plt.title("PCA 2D Final Feature Space dengan Penanda Outlier Multivariat")
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]*100:.2f}% var.)")
plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]*100:.2f}% var.)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

if df_vis["class"].nunique() > 1 and len(df_vis) > df_vis["class"].nunique():
    try:
        sil = silhouette_samples(X_scaled_df, df_vis["class"].astype(str))
        df_vis["silhouette_sample"] = sil
        print("Silhouette score global pada final feature set:", silhouette_score(X_scaled_df, df_vis["class"].astype(str)))
        df_vis[["path", "class", "silhouette_sample", "consensus_multivariate_outlier_score", "outlier_strength"]].to_csv(
            "descriptive_outlier_with_silhouette.csv",
            index=False,
        )
    except Exception as e:
        print("Silhouette sample tidak dapat dihitung:", e)


## 7. Tabel top outlier dan galeri citra

Bagian ini membantu memverifikasi apakah skor outlier multivariat masuk akal secara visual. Sampel tetap **tidak dihapus otomatis**.

In [ ]:

top_outliers = df_out.sort_values(
    ["consensus_multivariate_outlier_score", "mahalanobis_score", "lof_score", "pca_reconstruction_error"],
    ascending=[False, False, False, False],
).copy()

top_outlier_cols = [
    "path", "class", "consensus_multivariate_outlier_score", "outlier_strength",
    "mahalanobis_score", "lof_score", "isolation_forest_score", "pca_reconstruction_error",
] + flag_cols

display(top_outliers[top_outlier_cols].head(30))
top_outliers[top_outlier_cols].head(100).to_csv("top_descriptive_multivariate_outliers.csv", index=False)

# Galeri top outlier dengan file yang tersedia.
available = top_outliers[top_outliers["path"].apply(lambda p: Path(str(p)).exists())].head(12)

if len(available) == 0:
    print("Tidak ada path gambar yang tersedia untuk galeri. Cek kolom path pada final feature set.")
else:
    n = len(available)
    n_cols = 4
    n_rows = int(np.ceil(n / n_cols))
    plt.figure(figsize=(14, 3.8 * n_rows))
    for i, (_, row) in enumerate(available.iterrows(), 1):
        try:
            img = Image.open(row["path"])
            plt.subplot(n_rows, n_cols, i)
            plt.imshow(img, cmap="gray")
            plt.axis("off")
            plt.title(
                f"{row['class']}\nscore={row['consensus_multivariate_outlier_score']} | {row['outlier_strength']}",
                fontsize=9,
            )
        except Exception as e:
            print(f"Gagal membuka gambar {row['path']}: {e}")
    plt.suptitle("Galeri Top Descriptive Multivariate Outliers", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


## 8. Narasi interpretasi untuk laporan

Gunakan narasi berikut pada Bab IV/Bab V sesuai kebutuhan:

> Tahap post-processing outlier detection dilakukan setelah proses seleksi fitur dan Hybrid PCA. Berbeda dari analisis univariate pada tahap feature extraction, tahap ini menggunakan final feature set yang terdiri atas core stable features dan komponen PCA residual. Outlier dianalisis secara multivariat menggunakan Robust Mahalanobis Distance, Local Outlier Factor, Isolation Forest, dan PCA Reconstruction Error. Keempat metode tersebut tidak digunakan untuk menghapus data secara otomatis, tetapi untuk memberi skor deskriptif terhadap sampel yang memiliki karakter visual ekstrem relatif terhadap kelasnya. Dengan demikian, post-processing outlier detection berfungsi sebagai alat audit kualitas data dan analisis overlap antarkelas, bukan sebagai mekanisme eliminasi data.
